# Chat models and messages

Assistant version 1: Stream a support reply and report token usage and estimated cost.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --group authoring`, then `uv run jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## init_chat_model; invoke; switch OpenAI, Anthropic, Gemini, Ollama via configuration

Predict the output before running. The function definitions below match `examples/01_hello_model.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Use the provider-neutral factory built on init_chat_model.

Run: uv run python modules/01_chat_models/examples/01_hello_model.py --offline
Expected: One AIMessage and the selected provider name.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_text


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: get_chat_model delegates to init_chat_model in common/models.py.
    model = get_chat_model(settings=settings)
    print_text(f"Configured provider: {settings.llm_provider}")
    print_message(model.invoke("Greet a Brightwave customer in one sentence."))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## SystemMessage, HumanMessage, AIMessage, ToolMessage, plain-dict messages

Predict the output before running. The function definitions below match `examples/02_message_types.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Inspect message roles and a correctly matched tool-result transcript.

Run: uv run python modules/01_chat_models/examples/02_message_types.py --offline
Expected: system, human, ai, tool roles, followed by a reply from plain-dict input.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message


def example_history() -> list:
    return [
        SystemMessage("Help Brightwave customers."),
        HumanMessage("Stock for BW-001?"),
        AIMessage(
            content="",
            tool_calls=[{"name": "check_stock", "args": {"sku": "BW-001"}, "id": "stock-1"}],
        ),
        ToolMessage(content='{"stock": 12}', tool_call_id="stock-1"),
    ]


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: ToolMessage is a program result, not a new instruction from the customer.
    for message in example_history():
        print_message(message)
    print_message(
        get_chat_model(settings=settings).invoke(
            [
                {"role": "system", "content": "Keep Brightwave replies brief."},
                {"role": "user", "content": "Say hello."},
            ]
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## stream; assemble content chunks; print a reply as it arrives

Predict the output before running. The function definitions below match `examples/03_streaming.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Consume incremental message chunks and reconstruct a complete response.

Run: uv run python modules/01_chat_models/examples/03_streaming.py --offline
Expected: A streamed reply, then its character count.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_stream, print_text


def main() -> None:
    settings = lesson_settings()
    model = get_chat_model(
        settings=settings, responses=["Thanks for contacting Brightwave.\nHow can we help?"]
    )
    # KEY IDEA: each chunk is only a piece; downstream storage needs the assembled text.
    text = print_stream(model.stream("Greet a Brightwave customer in two short lines."))
    print_text(f"Assembled {len(text)} characters.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## batch, ainvoke; introduce async, await and asyncio before use

Predict the output before running. The function definitions below match `examples/04_batch_async.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Compare batch with async/await: concurrency overlaps waiting for responses.

Run: uv run python modules/01_chat_models/examples/04_batch_async.py --offline
Expected: Two ordered batch replies and one awaited reply.
Cost: Three short hosted calls, typically under $0.01 total; offline/local no API fee.
"""

import asyncio

from langchain_core.language_models import BaseChatModel

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message


async def greet_async(model: BaseChatModel) -> None:
    # await yields control while IO is pending; it does not create another paid request.
    print_message(await model.ainvoke("Welcome a Brightwave customer."))


def main() -> None:
    settings = lesson_settings()
    model = get_chat_model(settings=settings)
    for reply in model.batch(
        ["Greet a tent buyer.", "Greet a backpack buyer."], config={"max_concurrency": 2}
    ):
        print_message(reply)
    # KEY IDEA: asyncio owns the event loop in a script. In notebooks use await directly.
    asyncio.run(greet_async(model))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## temperature, max_tokens, timeout, max_retries; provider capability differences

Predict the output before running. The function definitions below match `examples/05_parameters.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Set token, timeout, retry and sampling limits centrally.

Run: uv run python modules/01_chat_models/examples/05_parameters.py --offline
Expected: Configured limits and a short reply; unsupported sampling parameters are omitted.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_text


def main() -> None:
    settings = lesson_settings().model_copy(
        update={"model_max_tokens": 96, "model_timeout": 20, "model_max_retries": 1}
    )
    # KEY IDEA: Ollama uses num_predict; hosted adapters use max_tokens in our factory.
    print_text("Output cap 96 tokens; timeout 20 seconds; at most 1 transport retry.")
    print_message(
        get_chat_model(settings=settings, temperature=0).invoke(
            "Write one Brightwave greeting. Do not make product or policy claims."
        )
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## usage_metadata; input, output and reasoning tokens; missing-usage handling; cost estimate

Predict the output before running. The function definitions below match `examples/06_usage_cost.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Read provider token usage, including optional reasoning-token details.

Run: uv run python modules/01_chat_models/examples/06_usage_cost.py --offline
Expected: Usage fields and an honest unknown estimate when usage is absent.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from langchain_core.messages import AIMessage

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text, print_usage


def main() -> None:
    settings = lesson_settings()
    fixture = AIMessage(
        content="Welcome to Brightwave.",
        usage_metadata={
            "input_tokens": 20,
            "output_tokens": 8,
            "total_tokens": 28,
            "output_token_details": {"reasoning": 2},
        },
    )
    reply = get_chat_model(settings=settings, responses=[fixture]).invoke("Greet a customer.")
    usage = reply.usage_metadata or {}
    print_text(str(usage))
    print_text(
        "Reasoning tokens: "
        f"{usage.get('output_token_details', {}).get('reasoning', 'not reported')}"
    )
    # KEY IDEA: reasoning details are a subset of output tokens; never count them twice.
    print_usage(reply, settings=settings)
    print_usage(AIMessage(content="No usage supplied"), settings=settings)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## content_blocks; image input; local vision-model capability check

Predict the output before running. The function definitions below match `examples/07_multimodal.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Send a local synthetic product image through standard content blocks.

Run: uv run python modules/01_chat_models/examples/07_multimodal.py --offline
Expected: A description of the simple tent drawing; never a claim about real product quality.
Cost: One image call; price depends on image tokens, typically cents; fixtures are free.
"""

import base64

import httpx
from langchain_core.messages import HumanMessage

from common.config import ConfigurationError
from common.data import data_path
from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_text


def main() -> None:
    settings = lesson_settings()
    if settings.llm_provider == "ollama" and not settings.course_offline:
        response = httpx.post(
            settings.ollama_base_url.rstrip("/") + "/api/show",
            json={"model": settings.ollama_vision_model},
            timeout=5,
        )
        response.raise_for_status()
        if "vision" not in response.json().get("capabilities", []):
            raise ConfigurationError(
                "Pull the vision model in .env; the selected model lacks vision."
            )
    encoded = base64.b64encode(data_path("fixtures/product.png").read_bytes()).decode("ascii")
    message = HumanMessage(
        content_blocks=[
            {
                "type": "text",
                "text": "Describe only visible shapes in this Brightwave tent illustration.",
            },
            {"type": "image", "base64": encoded, "mime_type": "image/png"},
        ]
    )
    print_text(f"Input block types: {[block['type'] for block in message.content_blocks]}")
    # KEY IDEA: a text-only model cannot infer image pixels from their filename.
    print_message(
        get_chat_model(
            settings=settings,
            vision=True,
            responses=["A teal triangular tent on a light background."],
        ).invoke(message.content and [message])
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## InMemoryRateLimiter; bounded requests

Predict the output before running. The function definitions below match `examples/08_rate_limit.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Attach an in-process token-bucket request limiter to a model.

Run: uv run python modules/01_chat_models/examples/08_rate_limit.py --offline
Expected: Two replies paced at no more than two requests per second.
Cost: Two short calls; fixture/local modes have no API fee.
"""

from langchain_core.rate_limiters import InMemoryRateLimiter

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message


def main() -> None:
    settings = lesson_settings()
    limiter = InMemoryRateLimiter(
        requests_per_second=2, check_every_n_seconds=0.05, max_bucket_size=1
    )
    model = get_chat_model(settings=settings)
    model.rate_limiter = limiter
    # KEY IDEA: concurrency caps simultaneous work; rate limiting caps starts over time.
    for message in model.batch(
        ["Greet a tent buyer.", "Greet a boot buyer."], config={"max_concurrency": 2}
    ):
        print_message(message)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Build it yourself: httpx provider HTTP call; mocked transport offline; compare LangChain

Predict the output before running. The function definitions below match `examples/09_http_by_hand.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Build an OpenAI or Ollama HTTP call, then use the central LangChain factory.

Run: uv run python modules/01_chat_models/examples/09_http_by_hand.py --offline
Expected: The mock raw API and LangChain both return a greeting in offline mode.
Cost: Two short hosted calls, typically under $0.01; local and offline have no API fee.
"""

import httpx

from common.config import ConfigurationError
from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_message, print_text


def mock_chat(request: httpx.Request) -> httpx.Response:
    if request.url.path == "/v1/responses":
        return httpx.Response(
            200,
            json={
                "status": "completed",
                "output": [
                    {
                        "type": "message",
                        "content": [{"type": "output_text", "text": "Welcome to Brightwave."}],
                    }
                ],
            },
        )
    return httpx.Response(
        200,
        json={"message": {"role": "assistant", "content": "Welcome to Brightwave."}, "done": True},
    )


def main() -> None:
    settings = lesson_settings()
    if settings.llm_provider not in {"openai", "ollama"}:
        raise ConfigurationError(
            "This wire-protocol lesson supports OpenAI and Ollama. "
            "Set LLM_PROVIDER=openai or ollama; other lessons use the common adapter."
        )
    transport = httpx.MockTransport(mock_chat) if settings.course_offline else None
    messages = [{"role": "user", "content": "Greet a Brightwave customer in one sentence."}]
    # KEY IDEA: by hand, we own endpoint, JSON, HTTP status, timeout and response parsing.
    with httpx.Client(
        transport=transport, timeout=settings.model_timeout, trust_env=False
    ) as client:
        if settings.llm_provider == "ollama":
            url = settings.ollama_base_url.rstrip("/") + "/api/chat"
            headers = {}
            payload = {
                "model": settings.chat_model,
                "messages": messages,
                "stream": False,
                "options": {"num_predict": 96, "temperature": 0},
            }
        else:
            url = "https://api.openai.com/v1/responses"
            headers = (
                {}
                if settings.course_offline
                else {"Authorization": "Bearer " + settings.api_key("openai")}
            )
            payload = {
                "model": settings.chat_model,
                "input": messages,
                "store": False,
                "max_output_tokens": settings.model_max_tokens,
                "reasoning": {"effort": settings.openai_reasoning_effort},
            }
        reply = client.post(url, headers=headers, json=payload)
        reply.raise_for_status()
        result = reply.json()
        if settings.llm_provider == "ollama":
            text = result["message"]["content"]
        else:
            if result.get("status") != "completed":
                raise ConfigurationError("Response incomplete; inspect the output-token budget.")
            text = "".join(
                block["text"]
                for item in result["output"]
                if item["type"] == "message"
                for block in item["content"]
                if block["type"] == "output_text"
            )
        print_text("Raw HTTP: " + text)
    print_message(
        get_chat_model(settings=settings, responses=["Welcome to Brightwave."]).invoke(messages)
    )


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Stream a support reply and report token usage and estimated cost

Predict the output before running. The function definitions below match `use_case/reply_drafter.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Stream a polite ticket reply grounded in a supplied Brightwave policy.

Run: uv run python modules/01_chat_models/use_case/reply_drafter.py --offline
Expected: A draft and usage/cost summary; no email is sent.
Cost: One short hosted call, typically under $0.01; --offline is free.
"""

from collections.abc import Iterator

from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage

from common.data import data_path, load_jsonl
from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_stream, print_usage


def main() -> None:
    settings = lesson_settings()
    ticket = load_jsonl("support_tickets.jsonl")[1]
    policy = data_path("policies/returns.md").read_text(encoding="utf-8")
    model = get_chat_model(settings=settings)
    total = None

    def chunks() -> Iterator[BaseMessage]:
        nonlocal total
        for chunk in model.stream(
            [
                SystemMessage(
                    "Draft only, using this policy. Never assert a refund was issued. " + policy
                ),
                HumanMessage(ticket["text"]),
            ]
        ):
            total = chunk if total is None else total + chunk
            yield chunk

    # KEY IDEA: adding chunks preserves final usage metadata as well as text.
    print_stream(chunks())
    if total is not None:
        print_usage(total, settings=settings)


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run pytest modules/01_chat_models/tests` for the supplied solution checks.